Introduction

This notebook explores the credit card fraud dataset before any modeling. The goal is to understand the data's shape, the class imbalance, which features carry fraud signal, and any data quality issues that need fixing before training. Findings are written after each cell based on actual output, not assumptions.

Load data

In [ ]:
import pandas as pd

df = pd.read_csv("../data/raw/creditcard.csv")

print(df.shape)
print(df.dtypes)
df.head()

Dataset has 284,807 rows and 31 columns.
Columns match what was expected: Time, V1 to V28, Amount, and Class.4
All columns loaded as numbers (float64 or int64), no text or broken values.



Class distribution — the core problem of this dataset

In [ ]:
class_counts = df["Class"].value_counts()
class_percent = df["Class"].value_counts(normalize=True) * 100

print(class_counts)
print(class_percent)

284,315 transactions are genuine, only 492 are fraud.
Fraud is 0.17% of the data.
This is a severely imbalanced dataset. A model that predicts "genuine" every time would be 99.8% accurate while catching zero fraud. Accuracy is not a usable metric here.

Time and Amount — the only two non-PCA features

In [ ]:
df[["Time", "Amount"]].describe()
df.groupby("Class")["Amount"].describe()

Fraud transactions average $122.21, genuine average $88.29 — fraud looks higher on average.
But the median tells a different story: fraud median is $9.25, genuine median is $22.00. So typical fraud transactions are actually smaller than typical genuine ones.
The average is misleading because a few large amounts pull it up. Amount alone is not a clean fraud signal, but it likely adds some value combined with other features.

Fraud pattern over time

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(df[df["Class"] == 0]["Time"], bins=48, alpha=0.6, label="Genuine")
axes[0].hist(df[df["Class"] == 1]["Time"], bins=48, alpha=0.6, label="Fraud")
axes[0].set_title("Time distribution (unscaled counts)")
axes[0].legend()

axes[1].hist(df[df["Class"] == 1]["Time"], bins=48, color="red")
axes[1].set_title("Fraud transactions over time")

plt.tight_layout()
plt.show()

Genuine transactions follow a clear day/night pattern — busy during the day, quiet overnight, repeating across the two days in the dataset.
Fraud does not follow that same pattern. Two spikes stand out (around Time 40,000 and 95,000), both landing near the overnight low in genuine activity.
This suggests fraud is relatively more common when normal transaction volume is low. With only 492 fraud cases spread across the timeline, this pattern is real but should not be treated as a strong rule.

 Correlation of V1-V28 with Class

In [ ]:
correlations = df.corr()["Class"].sort_values(ascending=False)
print(correlations)

No single feature strongly predicts fraud on its own. The strongest is V17 at -0.33, which is a weak-to-moderate correlation, not a dominant one.
The most fraud-linked features are V17, V14, V12, V10, V11, and V4.
Because no single feature separates fraud cleanly, the model needs to combine features nonlinearly. This supports using a tree-based model like XGBoost instead of a simple linear model.

Distributions of top correlated features by Class

In [ ]:
import matplotlib.pyplot as plt

top_features = ["V17", "V14", "V12", "V10", "V11", "V4"]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, feature in enumerate(top_features):
    axes[i].boxplot(
        [df[df["Class"] == 0][feature], df[df["Class"] == 1][feature]],
        labels=["Genuine", "Fraud"]
    )
    axes[i].set_title(feature)

plt.tight_layout()
plt.show()

For V17, V14, V12, and V10, fraud transactions sit noticeably lower than genuine ones, with little overlap in the middle ranges.
For V11 and V4, fraud transactions sit noticeably higher than genuine ones.
These six features show real, visible separation between fraud and genuine, confirming what the correlation numbers suggested. Still overlapping enough that no single feature can be used alone as a fraud rule.

Duplicates and missing values

In [ ]:
print("Missing values per column:")
print(df.isnull().sum().sum())

print("\nDuplicate rows:")
print(df.duplicated().sum())

Zero missing values across the entire dataset. No cleaning needed for missing data.
1,081 exact duplicate rows found. Given 28 continuous decimal columns plus Time and Amount, identical values across all of them are not a natural coincidence — these are recording duplicates, not real repeated transactions.

Correlation heatmap among V-features

In [ ]:
import seaborn as sns

v_features = [f"V{i}" for i in range(1, 29)]
corr_matrix = df[v_features].corr()

plt.figure(figsize=(14, 12))
sns.heatmap(corr_matrix, cmap="coolwarm", center=0, annot=False)
plt.title("Correlation among V1-V28")
plt.show()

No correlation at all between any pair of V1 to V28. This is expected since PCA components are mathematically built to be independent of each other.
Confirms there is no redundancy to remove among the V-features.

Check duplicate breakdown by Class, then drop

In [ ]:
duplicates = df[df.duplicated()]
print(duplicates["Class"].value_counts())

df = df.drop_duplicates()
print("\nNew shape after dropping duplicates:", df.shape)

Of the 1,081 duplicates, 1,062 were genuine transactions and 19 were fraud.
Removing them was necessary, not optional. Keeping duplicates risks the same transaction appearing in both the train and test sets later, which would let the model be evaluated on data it already memorized, making results look better than they really are.
After removal, dataset shrinks to 283,726 rows, with fraud count now 473. Fraud rate stays at roughly 0.167%, basically unchanged.

save the deduplicated data and the train/test split

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["Class"])
y = df["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

train_df = X_train.copy()
train_df["Class"] = y_train
test_df = X_test.copy()
test_df["Class"] = y_test

train_df.to_csv("../data/processed/train.csv", index=False)
test_df.to_csv("../data/processed/test.csv", index=False)

print("Train shape:", train_df.shape)
print("Train fraud count:", train_df["Class"].sum())
print("Test shape:", test_df.shape)
print("Test fraud count:", test_df["Class"].sum())

Data split 80/20 into train and test, using stratification so both sets keep the same fraud ratio.
Train set: 226,980 rows, 378 fraud cases.
Test set: 56,746 rows, 95 fraud cases.
This exact split is saved to disk and will be reused in every following notebook, so results across different models stay comparable.

Conclusion

This dataset is clean — no missing values, and the only issue found was 1,081 duplicate rows, which have been removed. The core challenge is severe class imbalance: fraud is only 0.17% of transactions, ruling out accuracy as a metric and requiring imbalance-aware training and evaluation using AUPRC. No single feature predicts fraud on its own, but six features (V17, V14, V12, V10, V11, V4) show real separation between classes and are expected to matter most to the model. Amount and Time do not directly separate fraud but show mild, usable patterns — fraud has a different amount distribution shape and shows some tendency to occur during low-activity hours. The data has been split into a fixed 80/20 train/test set with stratification, ready for supervised modeling in the next notebook.
